# Day 2 Session 3 -- Demos: LangGraph Orchestration & Workflow Design

This notebook contains the **instructor-led demos** for Session 3. Run each cell, observe the output, and follow along as the instructor explains each LangGraph pattern.

**Engineering context:** You are an engineer building workflow orchestration systems. Your users (consultants) need reliable, stateful pipelines that route work intelligently, refine outputs iteratively, and include human checkpoints. LangGraph lets you model these requirements as directed graphs with typed state, conditional edges, and cycles.

In [1]:
!pip install -q langchain langchain-openai langchain-core langgraph python-dotenv


[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: pip install --upgrade pip


## Setup

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["LANGCHAIN_TRACING_V2"] = os.getenv("LANGCHAIN_TRACING_V2", "false")
os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGCHAIN_PROJECT", "mckinsey-genai-3day")

from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from typing import TypedDict, Annotated, Literal
import operator
import json
import os

print("All imports successful!")

All imports successful!


---
## Demo 1: Linear Pipeline -- Modeling a Consulting Engagement Pipeline

A StateGraph has three core components:
- **State**: A TypedDict that holds all data flowing through the graph (the engagement tracker)
- **Nodes**: Python functions that read and update state (each node is a step in the pipeline)
- **Edges**: Connections that define the execution order (the workflow sequence)

Here we model a simple engagement intake pipeline: a client request arrives, we extract the scope, classify the industry sector, and format a preliminary engagement brief.

In [3]:
# Demo 1 - LangGraph Basics: Consulting Engagement Pipeline

# Step 1: Define the state schema (the engagement tracker)
class EngagementState(TypedDict):
    client_request: str
    scope_summary: str
    industry_sector: str
    engagement_brief: str

# Step 2: Define node functions (each takes state, returns partial state update)
def extract_scope_node(state: EngagementState) -> dict:
    """Extract and normalize the engagement scope from the client request."""
    return {"scope_summary": state["client_request"].strip().upper()}

def classify_industry_node(state: EngagementState) -> dict:
    """Classify the industry sector based on keywords in the request."""
    text = state["client_request"].lower()
    if any(kw in text for kw in ["bank", "financial", "fintech", "insurance"]):
        sector = "Financial Services"
    elif any(kw in text for kw in ["pharma", "health", "biotech", "hospital"]):
        sector = "Healthcare & Life Sciences"
    elif any(kw in text for kw in ["retail", "consumer", "e-commerce", "cpg"]):
        sector = "Consumer & Retail"
    else:
        sector = "Cross-Industry"
    return {"industry_sector": sector}

def format_brief_node(state: EngagementState) -> dict:
    """Format the preliminary engagement brief."""
    return {"engagement_brief": f"ENGAGEMENT BRIEF\nScope: {state['scope_summary']}\nSector: {state['industry_sector']}"}

# Step 3: Build the graph
graph = StateGraph(EngagementState)

graph.add_node("extract_scope", extract_scope_node)
graph.add_node("classify_industry", classify_industry_node)
graph.add_node("format_brief", format_brief_node)

graph.set_entry_point("extract_scope")
graph.add_edge("extract_scope", "classify_industry")
graph.add_edge("classify_industry", "format_brief")
graph.add_edge("format_brief", END)

# Step 4: Compile and run
app = graph.compile()

result = app.invoke({
    "client_request": "Our fintech startup needs help with market entry strategy for Southeast Asia",
    "scope_summary": "", "industry_sector": "", "engagement_brief": ""
})
print("Engagement Pipeline Result:")
print(f"  Client Request  : {result['client_request']}")
print(f"  Scope Summary   : {result['scope_summary']}")
print(f"  Industry Sector : {result['industry_sector']}")
print(f"  Brief           : {result['engagement_brief']}")

Engagement Pipeline Result:
  Client Request  : Our fintech startup needs help with market entry strategy for Southeast Asia
  Scope Summary   : OUR FINTECH STARTUP NEEDS HELP WITH MARKET ENTRY STRATEGY FOR SOUTHEAST ASIA
  Industry Sector : Financial Services
  Brief           : ENGAGEMENT BRIEF
Scope: OUR FINTECH STARTUP NEEDS HELP WITH MARKET ENTRY STRATEGY FOR SOUTHEAST ASIA
Sector: Financial Services


---
## Demo 2: Conditional Routing -- Routing Client Engagements by Complexity

Conditional edges let the graph take different paths based on the current state. This is how workflows adapt -- a quick benchmarking study follows a different path than a full-scale transformation program.

Here we use an LLM to classify engagement complexity and route to the appropriate workstream: **rapid assessment**, **standard engagement**, or **transformation program**.

In [4]:
# Demo 2 - Conditional Edges: Routing Engagements by Complexity

class EngagementRouterState(TypedDict):
    engagement_request: str
    complexity: str
    workstream_output: str

llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0")))

def assess_complexity_node(state: EngagementRouterState) -> dict:
    """Use LLM to classify engagement complexity."""
    response = llm.invoke([
        SystemMessage(content="You are a McKinsey engagement manager. Classify this client request into exactly one category: 'rapid' (2-4 week benchmarking or quick scan), 'standard' (8-12 week strategy or operations engagement), or 'transformation' (6+ month large-scale change program). Respond with just the one word."),
        HumanMessage(content=state["engagement_request"])
    ])
    complexity = response.content.strip().lower()
    print(f"  Complexity assessed: {complexity}")
    return {"complexity": complexity}

def rapid_assessment_node(state: EngagementRouterState) -> dict:
    response = llm.invoke([
        SystemMessage(content="You are a McKinsey associate conducting a rapid assessment. Provide a concise 2-week diagnostic framework with key hypotheses to test."),
        HumanMessage(content=state["engagement_request"])
    ])
    return {"workstream_output": f"[RAPID ASSESSMENT]\n{response.content}"}

def standard_engagement_node(state: EngagementRouterState) -> dict:
    response = llm.invoke([
        SystemMessage(content="You are a McKinsey engagement manager designing a standard 10-week engagement. Outline the workplan with key phases: diagnostic, analysis, recommendations, and implementation roadmap."),
        HumanMessage(content=state["engagement_request"])
    ])
    return {"workstream_output": f"[STANDARD ENGAGEMENT]\n{response.content}"}

def transformation_node(state: EngagementRouterState) -> dict:
    response = llm.invoke([
        SystemMessage(content="You are a McKinsey senior partner designing a large-scale transformation program. Outline the multi-phase approach including change management, capability building, and performance tracking."),
        HumanMessage(content=state["engagement_request"])
    ])
    return {"workstream_output": f"[TRANSFORMATION PROGRAM]\n{response.content}"}

def route_by_complexity(state: EngagementRouterState) -> str:
    c = state["complexity"]
    if "rapid" in c:
        return "rapid_assessment"
    elif "transformation" in c:
        return "transformation"
    else:
        return "standard_engagement"

graph = StateGraph(EngagementRouterState)
graph.add_node("assess_complexity", assess_complexity_node)
graph.add_node("rapid_assessment", rapid_assessment_node)
graph.add_node("standard_engagement", standard_engagement_node)
graph.add_node("transformation", transformation_node)

graph.set_entry_point("assess_complexity")
graph.add_conditional_edges("assess_complexity", route_by_complexity, {
    "rapid_assessment": "rapid_assessment",
    "standard_engagement": "standard_engagement",
    "transformation": "transformation"
})
graph.add_edge("rapid_assessment", END)
graph.add_edge("standard_engagement", END)
graph.add_edge("transformation", END)

app = graph.compile()

requests = [
    "We need a quick competitive benchmarking of our pricing vs. top 3 rivals",
    "Help us design a new go-to-market strategy for entering the European healthcare market",
    "We are a $20B conglomerate and need to completely restructure our operating model across 15 business units"
]

for req in requests:
    print(f"\nRequest: {req}")
    result = app.invoke({"engagement_request": req, "complexity": "", "workstream_output": ""})
    print(f"Output: {result['workstream_output'][:200]}...\n{'='*60}")


Request: We need a quick competitive benchmarking of our pricing vs. top 3 rivals
  Complexity assessed: rapid
Output: [RAPID ASSESSMENT]
### 2-Week Diagnostic Framework for Competitive Pricing Benchmarking

#### Objective:
To assess and compare the pricing strategies of our company against the top three competitors i...

Request: Help us design a new go-to-market strategy for entering the European healthcare market
  Complexity assessed: standard
Output: [STANDARD ENGAGEMENT]
Designing a new go-to-market strategy for entering the European healthcare market requires a structured approach. Below is a detailed 10-week workplan divided into key phases: di...

Request: We are a $20B conglomerate and need to completely restructure our operating model across 15 business units
  Complexity assessed: transformation
Output: [TRANSFORMATION PROGRAM]
Designing a large-scale transformation program for a $20 billion conglomerate requires a structured, multi-phase approach that encompasses change m

---
## Demo 3: ReAct Agent -- McKinsey Market Research Analyst

The ReAct (Reason + Act) pattern models how a research analyst works:
1. **Think** -- Reason about what data is needed
2. **Act** -- Search for intelligence using available tools
3. **Observe** -- Process the findings
4. **Repeat** until a well-supported answer emerges

Our analyst has access to simulated tools for market data, financial metrics, and competitive intelligence.

In [5]:
# Demo 3 - ReAct Agent: McKinsey Market Research Analyst

class ReActState(TypedDict):
    question: str
    thoughts: list[str]
    actions: list[str]
    observations: list[str]
    final_answer: str
    step_count: int

llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0")))

# Simulated McKinsey research tools
def research_tool(query):
    """Simulated market intelligence database."""
    data = {
        "ev market size": "The global EV market was valued at $388B in 2024, projected to reach $950B by 2030 (CAGR ~16%). China leads with 60% market share.",
        "tesla competitors": "Key Tesla competitors: BYD (largest global EV seller by volume), Volkswagen Group (ID. series), Hyundai-Kia (fastest growing in Europe), and Rivian (US truck segment).",
        "ev margins": "Average EV gross margins: Tesla ~18%, BYD ~20%, legacy OEMs ~5-8% on EVs. Battery costs represent 30-40% of vehicle cost.",
        "southeast asia market": "Southeast Asia EV penetration is ~5% (2024), led by Thailand (govt subsidies) and Indonesia (nickel supply chain). Market expected to grow 25% CAGR through 2030.",
        "mckinsey frameworks": "Key McKinsey frameworks: MECE (Mutually Exclusive, Collectively Exhaustive), 7S Model, Three Horizons, Profit Pools, and Value Chain Analysis."
    }
    for key, val in data.items():
        if key in query.lower():
            return val
    return f"No data found for: {query}"

def think_node(state: ReActState) -> dict:
    """Analyst reasons about what data to gather next."""
    context = ""
    for i in range(len(state["actions"])):
        context += f"Action: {state['actions'][i]}\nObservation: {state['observations'][i]}\n"
    
    response = llm.invoke([
        SystemMessage(content="You are a McKinsey research analyst. Given the question and any previous research findings, decide your next step. Either search for more data (respond: ACTION: search '<query>') or provide a final synthesized answer (respond: FINAL ANSWER: <answer>). Think like a consultant -- be MECE in your approach."),
        HumanMessage(content=f"Research question: {state['question']}\n\n{context}\nWhat should I research next?")
    ])
    thought = response.content
    print(f"  Analyst thinks: {thought[:120]}...")
    return {"thoughts": state["thoughts"] + [thought]}

def act_node(state: ReActState) -> dict:
    """Execute the research action."""
    latest_thought = state["thoughts"][-1]
    
    if "FINAL ANSWER:" in latest_thought:
        answer = latest_thought.split("FINAL ANSWER:")[-1].strip()
        return {"final_answer": answer, "step_count": state["step_count"] + 1}
    
    if "ACTION: search" in latest_thought:
        query = latest_thought.split("search")[-1].strip().strip("'\"")
        observation = research_tool(query)
        print(f"  Research: search('{query}') -> {observation[:80]}...")
        return {
            "actions": state["actions"] + [f"search('{query}')"],
            "observations": state["observations"] + [observation],
            "step_count": state["step_count"] + 1
        }
    
    return {"final_answer": latest_thought, "step_count": state["step_count"] + 1}

def should_continue(state: ReActState) -> str:
    if state["final_answer"] or state["step_count"] >= 5:
        return "end"
    return "think"

graph = StateGraph(ReActState)
graph.add_node("think", think_node)
graph.add_node("act", act_node)

graph.set_entry_point("think")
graph.add_edge("think", "act")
graph.add_conditional_edges("act", should_continue, {"think": "think", "end": END})

app = graph.compile()

result = app.invoke({
    "question": "What is the EV market size and who are Tesla's main competitors?",
    "thoughts": [], "actions": [], "observations": [],
    "final_answer": "", "step_count": 0
})

print(f"\nAnalyst's Answer: {result['final_answer']}")
print(f"Research steps taken: {result['step_count']}")

  Analyst thinks: ACTION: search 'EV market size 2023'  
ACTION: search 'Tesla main competitors in EV market 2023'...
  Research: search('Tesla main competitors in EV market 2023') -> No data found for: Tesla main competitors in EV market 2023...
  Analyst thinks: ACTION: search 'EV market size 2023'...
  Research: search('EV market size 2023') -> The global EV market was valued at $388B in 2024, projected to reach $950B by 20...
  Analyst thinks: ACTION: search 'Tesla competitors in EV market 2023'...
  Research: search('Tesla competitors in EV market 2023') -> Key Tesla competitors: BYD (largest global EV seller by volume), Volkswagen Grou...
  Analyst thinks: ACTION: search('Tesla market share in EV market 2023')...
  Research: search('('Tesla market share in EV market 2023')') -> No data found for: ('Tesla market share in EV market 2023')...
  Analyst thinks: ACTION: search('Tesla market share in global EV market 2023')...
  Research: search('('Tesla market share in global EV marke

---
## Demo 4: Iterative Refinement -- Refining a Recommendation Until Partner-Quality

Cycles allow a graph to loop back and improve its output. Deliverables go through multiple rounds of refinement before they reach partner-quality. This demo models that iterative process: draft a strategic recommendation, have a simulated "partner review" critique it, and loop back to refine until it meets the bar or reaches max iterations.

In [6]:
# Demo 4 - Cycles: Refining a Recommendation Until Partner-Quality

class RecommendationState(TypedDict):
    strategic_question: str
    draft_recommendation: str
    partner_feedback: str
    iteration: int
    is_partner_quality: bool

llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0.7")))

def formulate_recommendation(state: RecommendationState) -> dict:
    """Draft or refine a strategic recommendation."""
    if state["draft_recommendation"]:
        prompt = f"You are a McKinsey engagement manager. Refine this strategic recommendation based on partner feedback. Make it more MECE, insight-driven, and actionable.\n\nCurrent recommendation: {state['draft_recommendation']}\n\nPartner feedback: {state['partner_feedback']}\n\nProvide an improved recommendation (3-4 sentences, structured and concise):"
    else:
        prompt = f"You are a McKinsey engagement manager. Draft a strategic recommendation for this question (3-4 sentences). Be specific, actionable, and data-driven.\n\nStrategic question: {state['strategic_question']}"
    
    response = llm.invoke([HumanMessage(content=prompt)])
    print(f"  [Iteration {state['iteration'] + 1}] Recommendation: {response.content[:100]}...")
    return {"draft_recommendation": response.content, "iteration": state["iteration"] + 1}

def partner_review(state: RecommendationState) -> dict:
    """Simulated partner review -- critiques the recommendation for quality."""
    response = llm.invoke([
        SystemMessage(content="You are a McKinsey senior partner reviewing a strategic recommendation. Rate it 1-10 on: (1) MECE structure, (2) actionability, (3) insight depth. If average is 8+, respond with 'APPROVED - ready for client'. Otherwise, provide specific feedback on what to improve."),
        HumanMessage(content=state["draft_recommendation"])
    ])
    feedback = response.content
    is_approved = "APPROVED" in feedback.upper() or state["iteration"] >= 3
    print(f"  [Partner Review] {'PARTNER APPROVED' if is_approved else feedback[:80]}...")
    return {"partner_feedback": feedback, "is_partner_quality": is_approved}

def should_refine(state: RecommendationState) -> str:
    if state["is_partner_quality"]:
        return "end"
    return "formulate_recommendation"

graph = StateGraph(RecommendationState)
graph.add_node("formulate_recommendation", formulate_recommendation)
graph.add_node("partner_review", partner_review)

graph.set_entry_point("formulate_recommendation")
graph.add_edge("formulate_recommendation", "partner_review")
graph.add_conditional_edges("partner_review", should_refine, {
    "formulate_recommendation": "formulate_recommendation", "end": END
})

app = graph.compile()

result = app.invoke({
    "strategic_question": "Should our client, a mid-size European bank, acquire a fintech payments startup to accelerate digital transformation?",
    "draft_recommendation": "", "partner_feedback": "", "iteration": 0, "is_partner_quality": False
})

print(f"\nFinal Recommendation (after {result['iteration']} iterations):")
print(result["draft_recommendation"])

  [Iteration 1] Recommendation: Based on our analysis, we recommend that the mid-size European bank pursue the acquisition of the fi...
  [Partner Review] (1) MECE structure: 7  
The recommendation is generally clear, but it could bene...
  [Iteration 2] Recommendation: **Strategic Recommendation:**

We recommend that the mid-size European bank pursue the acquisition o...
  [Partner Review] (1) MECE structure: 8  
(2) Actionability: 7  
(3) Insight depth: 7  

**Average...
  [Iteration 3] Recommendation: **Strategic Recommendation:**

We recommend that the mid-size European bank pursue the acquisition o...
  [Partner Review] PARTNER APPROVED...

Final Recommendation (after 3 iterations):
**Strategic Recommendation:**

We recommend that the mid-size European bank pursue the acquisition of the fintech payments startup to accelerate its digital transformation, structured around three distinct pillars:

1. **Rationale for Acquisition**: The startup's 30% year-over-year growth in transactio

---
## Demo 5: Human-in-the-Loop -- Partner Sign-Off Before Client Delivery

No major deliverable goes to the client without partner approval. LangGraph supports this through interrupt points where the graph pauses and waits for human approval before continuing. This demo models that gate: the system generates a client-ready action plan, pauses for partner review (simulated), and only proceeds to "deliver to client" if approved.

In [7]:
# Demo 5 - Human-in-the-Loop: Partner Sign-Off Before Client Delivery

class ClientDeliveryState(TypedDict):
    engagement_context: str
    action_plan: str
    partner_approved: bool
    delivery_output: str

llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0")))

def prepare_deliverable(state: ClientDeliveryState) -> dict:
    """Generate a client-ready action plan."""
    response = llm.invoke([
        SystemMessage(content="You are a McKinsey engagement manager. Create a structured 3-step action plan for the client. Include specific timelines, owners, and expected impact for each step."),
        HumanMessage(content=state["engagement_context"])
    ])
    print(f"  Deliverable prepared: {response.content[:120]}...")
    return {"action_plan": response.content}

def partner_signoff_node(state: ClientDeliveryState) -> dict:
    """Simulate partner review gate (auto-approve for demo)."""
    print(f"  [PARTNER REVIEW] Reviewing action plan for client delivery...")
    print(f"    Plan preview: {state['action_plan'][:200]}...")
    # In production, this would pause and wait for actual partner input
    approved = True  # Simulated approval
    print(f"  [PARTNER] {'Approved for client delivery' if approved else 'Needs revision -- send back to team'}")
    return {"partner_approved": approved}

def deliver_to_client(state: ClientDeliveryState) -> dict:
    """Package and deliver the final output to the client."""
    if not state["partner_approved"]:
        return {"delivery_output": "Delivery blocked -- partner approval required."}
    
    response = llm.invoke([
        SystemMessage(content="You are a McKinsey partner delivering recommendations to a C-suite client. Wrap this action plan in a professional executive summary with a compelling opening and clear next steps."),
        HumanMessage(content=f"Action Plan:\n{state['action_plan']}")
    ])
    return {"delivery_output": response.content}

def route_after_review(state: ClientDeliveryState) -> str:
    return "deliver_to_client" if state["partner_approved"] else "end"

graph = StateGraph(ClientDeliveryState)
graph.add_node("prepare_deliverable", prepare_deliverable)
graph.add_node("partner_signoff", partner_signoff_node)
graph.add_node("deliver_to_client", deliver_to_client)

graph.set_entry_point("prepare_deliverable")
graph.add_edge("prepare_deliverable", "partner_signoff")
graph.add_conditional_edges("partner_signoff", route_after_review, {
    "deliver_to_client": "deliver_to_client", "end": END
})
graph.add_edge("deliver_to_client", END)

app = graph.compile()

result = app.invoke({
    "engagement_context": "A Fortune 500 CPG company needs to reduce supply chain costs by 15% within 18 months while maintaining service levels",
    "action_plan": "", "partner_approved": False, "delivery_output": ""
})

print(f"\nClient Delivery:\n{result['delivery_output'][:300]}...")

  Deliverable prepared: **Action Plan to Reduce Supply Chain Costs by 15% within 18 Months**

**Step 1: Conduct a Comprehensive Supply Chain Ass...
  [PARTNER REVIEW] Reviewing action plan for client delivery...
    Plan preview: **Action Plan to Reduce Supply Chain Costs by 15% within 18 Months**

**Step 1: Conduct a Comprehensive Supply Chain Assessment**  
- **Timeline:** Months 1-3  
- **Owner:** Supply Chain Director and ...
  [PARTNER] Approved for client delivery

Client Delivery:
**Executive Summary**

In today’s competitive landscape, optimizing supply chain operations is not just a strategic advantage; it is a necessity for sustaining profitability and growth. Our analysis indicates that your organization has the potential to reduce supply chain costs by 15% within the nex...


---
## Summary

This demo notebook covered five core LangGraph patterns:

1. **Linear Pipeline** -- Typed state, nodes as functions, edges for sequential flow
2. **Conditional Routing** -- Graphs that take different paths based on LLM decisions
3. **ReAct Agent** -- Reason-Act-Observe loop as a cyclic graph for tool-using agents
4. **Iterative Refinement** -- Cycles that improve output quality over multiple passes
5. **Human-in-the-Loop** -- Pause points for human oversight before critical actions